# Last-token trigram predictor (`eta_generation = 0`)

This notebook estimates the simple sentence-final trigram distribution

$$p(x_{d-1} \mid x_{d-3}, x_{d-2})$$

from generated samples. It uses no inside-outside calculation and no parse information. Only the final three terminals of each sentence enter the counts. For the present purely ternary depth-3 grammar, every sentence has length 27.

The reported quantity is the direct plug-in conditional entropy of the sentence-final trigram distribution.

In [ ]:
import math
import random
import numpy as np

from Inside_outside import (
    generate_grammar,
    generate_rules_double_new,
    sample_from_grammar,
)

In [ ]:
# Grammar and sampling settings
V = 8
F = 1 / 4
DEPTH = 3
RULE_SEED = 12345678
SAMPLE_SEED = 20260824
ETA_GENERATION = 0
N_SAMPLES = 50_000

m_2 = int(F * V)
m_3 = int(F * V**2)
p_3 = 1 / (m_2 * ETA_GENERATION + m_3)
p_2 = ETA_GENERATION * p_3
p_2, p_3

In [ ]:
# Generate the fixed grammar, then use a separate reproducible sampling seed.
generation_rules, _ = generate_rules_double_new(
    V, V, m_2, m_3, p_2, p_3, DEPTH, RULE_SEED
)
grammar = generate_grammar(generation_rules)

random.seed(SAMPLE_SEED)
triples = np.empty((N_SAMPLES, 3), dtype=np.int64)
lengths = np.empty(N_SAMPLES, dtype=np.int64)

for sample_idx in range(N_SAMPLES):
    start_symbol = random.randint(1, V)
    sequence, _ = sample_from_grammar(
        grammar, str(start_symbol), DEPTH, p_2, p_3
    )
    lengths[sample_idx] = len(sequence)
    # Leaf symbols are numbered globally by grammar layer (25--32 here).
    # Map them back to zero-based terminal vocabulary indices 0--7.
    triples[sample_idx] = [(int(token) - 1) % V for token in sequence[-3:]]

print(f'Samples: {N_SAMPLES:,}')
print('Sentence lengths and counts:', dict(zip(*np.unique(lengths, return_counts=True))))
print('First five zero-based final triples:', triples[:5])

## Direct conditional-entropy estimate

For each context $(a,b)$, count its possible final tokens $c$. The reported conditional entropy is

$$H(C\mid A,B)=\sum_{a,b} \hat p(a,b) H(\hat p(C\mid a,b)).$$

This is the entropy/loss of the empirical trigram distribution evaluated on the same distribution. The corresponding deterministic trigram classifier predicts $\arg\max_c \hat p(c\mid a,b)$; its plug-in top-1 accuracy is $\sum_{a,b}\hat p(a,b)\max_c\hat p(c\mid a,b)$.

In [ ]:
def trigram_counts(final_triples, vocabulary_size=V):
    counts = np.zeros((vocabulary_size, vocabulary_size, vocabulary_size), dtype=np.int64)
    np.add.at(counts, tuple(final_triples.T), 1)
    return counts


def conditional_entropy_from_counts(counts):
    context_counts = counts.sum(axis=2, keepdims=True)
    probabilities = np.divide(
        counts, context_counts,
        out=np.zeros_like(counts, dtype=float),
        where=context_counts > 0,
    )
    log_probabilities = np.zeros_like(probabilities)
    np.log(probabilities, out=log_probabilities, where=probabilities > 0)
    context_entropies = -(probabilities * log_probabilities).sum(axis=2)
    context_weights = context_counts[..., 0] / counts.sum()
    return float((context_weights * context_entropies).sum())


counts = trigram_counts(triples)
entropy_nats = conditional_entropy_from_counts(counts)
entropy_bits = entropy_nats / math.log(2)
entropy_normalized = entropy_nats / math.log(V)
trigram_accuracy = counts.max(axis=2).sum() / counts.sum()

print(f'Empirical H(last | previous two): {entropy_nats:.6f} nats')
print(f'                                 {entropy_bits:.6f} bits')
print(f'                                 {entropy_normalized:.6f} normalized by log(V)')
print(f'Empirical trigram top-1 accuracy: {trigram_accuracy:.6f} ({trigram_accuracy:.3%})')
print(f'Uniform-guess accuracy:            {1 / V:.6f} ({1 / V:.3%})')
print(f'Observed contexts: {(counts.sum(axis=2) > 0).sum()} / {V**2}')

In [ ]:
import numpy as np
0.590867*np.log(8)/np.log(16)

In [ ]:
def trigram_counts(final_triples, vocabulary_size=V):
    counts = np.zeros((vocabulary_size, vocabulary_size, vocabulary_size), dtype=np.int64)
    np.add.at(counts, tuple(final_triples.T), 1)
    return counts


def conditional_entropy_from_counts(counts):
    context_counts = counts.sum(axis=2, keepdims=True)
    probabilities = np.divide(
        counts, context_counts,
        out=np.zeros_like(counts, dtype=float),
        where=context_counts > 0,
    )
    log_probabilities = np.zeros_like(probabilities)
    np.log(probabilities, out=log_probabilities, where=probabilities > 0)
    context_entropies = -(probabilities * log_probabilities).sum(axis=2)
    context_weights = context_counts[..., 0] / counts.sum()
    return float((context_weights * context_entropies).sum())


counts = trigram_counts(triples)
entropy_nats = conditional_entropy_from_counts(counts)
entropy_bits = entropy_nats / math.log(2)
entropy_normalized = entropy_nats / math.log(V)
trigram_accuracy = counts.max(axis=2).sum() / counts.sum()

print(f'Empirical H(last | previous two): {entropy_nats:.6f} nats')
print(f'                                 {entropy_bits:.6f} bits')
print(f'                                 {entropy_normalized:.6f} normalized by log(V)')
print(f'Empirical trigram top-1 accuracy: {trigram_accuracy:.6f} ({trigram_accuracy:.3%})')
print(f'Uniform-guess accuracy:            {1 / V:.6f} ({1 / V:.3%})')
print(f'Observed contexts: {(counts.sum(axis=2) > 0).sum()} / {V**2}')